imports

In [22]:
import pandas as pd 
import numpy as np
import cv2 
import os
import chromadb
from insightface.app import FaceAnalysis

In [9]:
model = FaceAnalysis(name="buffalo_l", providers=['CUDAExecutionProvider'])
model.prepare(ctx_id=0, det_size=(640, 640))

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/arc/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], wi

In [10]:
chroma_client = chromadb.PersistentClient(path = "sdf")
collection = chroma_client.get_or_create_collection(name="face_embeddings",
                                                    metadata={"hnsw:space": "cosine"})

In [11]:
import uuid
import shutil

Quality filter

In [23]:
def quality(img,face,blur,conf,ratio):
    if face.det_score < conf:
        return False
    bbox = face.bbox.astype(int)
    x1, y1, x2, y2 = bbox
    y1,y2 = max(0,y1), min(img.shape[0],y2)
    x1,x2 = max(0,x1), min(img.shape[1],x2)
    face_crop = img[y1:y2, x1:x2]
    if face_crop.size == 0:
        return False
    
    img_area = img.shape[0] * img.shape[1]
    face_area = (x2 - x1) * (y2 - y1)
    ratios = face_area / img_area
    if ratios < ratio:
        return False
    
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if laplacian_var < blur:
        return False
    return True

same dirs as before

In [24]:
dump = "/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb"
output = "/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/facial_output"
os.makedirs(output, exist_ok=True)
skipp = os.path.join(output,"skipped")
os.makedirs(skipp,exist_ok=True)

In [52]:
blur  = 80
conf = 0.8
ratio = 0.04

In [53]:
all_embed = []
for folder in os.listdir(dump):
    for img_name in os.listdir(os.path.join(dump,folder)):
        img_path = os.path.join(os.path.join(dump,folder),img_name)
        img = cv2.imread(img_path)
        faces = model.get(img)
        for face in faces:
            blur_check = quality(img,face,blur,conf,ratio)
            if not blur_check:
                continue
            all_embed.append((img_path,face.embedding.tolist(),img_path))


In [54]:
print(len(all_embed))

66
